# Python Companion

The lectures use Python without stopping to explain it: class time goes to the
finance, and the AI writes most of the code. This notebook is where the code
gets explained.

It is not a Python course. It covers only what actually appears in the lecture
notebooks, in the order it first appears, and every entry points back to where
it was used. When a line in a lecture looks like magic, look it up here.

Each entry has three parts:

- **The line**, exactly as it appears in class
- **What it does**, piece by piece, usually on a table small enough to see all of
- **📍 Used in**, the lecture and section, linked

Run the Setup cell first, then the entries in order — later entries use the
data loaded by earlier ones. The code cells are short on purpose. Change
something and re-run: that is the fastest way to find out what a line does.

## 📋 Contents

**From *The Workflow, and What a Return Is***

1. [Libraries, and the Setup cell](#imports)
2. [Variables and f-strings](#fstrings)
3. [Calling a function: `yf.download`, `.iloc`](#yfinance)
4. [Fama-French from the internet: `web.DataReader`](#datareader)
5. [Fixing the dates: `to_timestamp` and `MonthEnd`](#monthend)
6. [Columns: `ff['RF']` and `ff[['Mkt-RF', 'RF']]`](#columns)
7. [The course panel: `pd.read_parquet`](#parquet)
8. [One stock out of the panel: masks and `set_index`](#onestock)
9. [Whole-column arithmetic: `.prod()`, `.mean()`, `.std()`](#vector)
10. [Lining up two series by date: `.reindex`, `.dropna`](#align)
11. [The memo and the submission cell](#submission)

**From *The Panel and Portfolio Mathematics***

12. [The panel is long](#long)
13. [`scipy.stats`: skewness and kurtosis](#scipy)
14. [A figure: `fig, ax = plt.subplots()`](#figure)
15. [`groupby`, part one: one number per stock](#groupby1)
16. [`.rename`, `.idxmin`, `.strftime`](#idxmin)
17. [`.shift(1)`: last month's value](#shift)
18. [`groupby('permno')['me'].shift(1)`: last month's value, per stock](#groupshift)
19. [One month's portfolio by hand: weights, `@`, `np.average`](#weights)
20. [`groupby`, part two: one number per month, and `lambda`](#groupby2)
21. [Growth of $1 on a log scale](#logscale)

---

## 🛠️ Setup

In [ ]:
#@title Setup — run this first
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = [11, 4.5]
plt.rcParams['font.size'] = 11
import warnings; warnings.filterwarnings('ignore')

print("✅ Ready")

---

## From *The Workflow, and What a Return Is*

### 1. Libraries, and the Setup cell <a id="imports"></a>

> ```python
> import numpy as np
> import pandas as pd
> import matplotlib.pyplot as plt
> ```

Python on its own knows arithmetic, text and lists. Almost everything else
comes from **libraries** — code other people wrote, which you load with
`import`. Three do nearly all the work in this course:

| Library | What it is for | You will see |
|---|---|---|
| `numpy` | numbers and arrays: square roots, weighted averages | `np.sqrt(12)`, `np.average(...)` |
| `pandas` | tables with labelled rows and columns | `pd.read_parquet(...)`, `panel.groupby(...)` |
| `matplotlib` | figures | `plt.subplots()`, `ax.plot(...)` |

`as np` is a nickname. Without it you would type `numpy.sqrt(12)` every time;
with it, `np.sqrt(12)`. The nicknames `np`, `pd` and `plt` are universal —
every example online and everything the AI writes uses them.

The rest of the Setup cell is cosmetic:

- `%matplotlib inline` is not Python. It is a notebook command that puts each
  figure under the cell that made it.
- `plt.style.use(...)` and `plt.rcParams[...]` set how figures look by default.
- `warnings.filterwarnings('ignore')` hides warning messages. That keeps the
  output clean, and it also hides warnings you might have wanted. If something
  behaves strangely, delete that line and re-run.
- `#@title` is a Colab label that names the cell and lets you collapse it. To
  Python it is a comment, like anything else after `#`.

📍 **Used in:** the Setup cell of every lecture, starting with
[*The Workflow, and What a Return Is*](https://amoreira2.github.io/UG54/chapters/Finance/L1_Welcome_Returns_AI.html#setup-cell).

In [ ]:
print(np.sqrt(12))                       # numpy: a square root
print(pd.Series([0.10, -0.05, 0.02]))    # pandas: a labelled column of numbers

### 2. Variables and f-strings <a id="fstrings"></a>

> ```python
> STOCK      = "AAPL"
> INVESTMENT = 1000
> print(f"${INVESTMENT:,} in {STOCK} from {BIRTH_DATE} to {END_DATE}")
> ```

`=` stores a value under a name. `"AAPL"` in quotes is text (a *string*);
`1000` without quotes is a number. The capital letters are a convention for
"settings you are meant to edit", nothing more.

The `f` before the quotes makes an **f-string**: anything inside `{ }` is
replaced by its value, and after a colon you can say how to format it. Nearly
every `print` in the course uses one. The codes worth knowing:

| Code | Does | with `x = 0.08443` |
|---|---|---|
| `{x}` | as is | `0.08443` |
| `{x:.2f}` | 2 decimals | `0.08` |
| `{x:.1%}` | percent, 1 decimal | `8.4%` |
| `{x:+.2%}` | always show the sign | `+8.44%` |
| `{x:>9.1%}` | right-aligned in 9 characters | `     8.4%` |
| `{n:,}` | thousands separator | `85,428` |

Dates have their own codes: `{d:%Y-%m}` gives `1995-06`, `{d:%B %Y}` gives
`June 1995`.

📍 **Used in:** [Live Demo, Step 1 — Specify](https://amoreira2.github.io/UG54/chapters/Finance/L1_Welcome_Returns_AI.html#step-1-specify), and every
`print` after it.

In [ ]:
STOCK      = "AAPL"
INVESTMENT = 1000
x = 0.08443

print(f"${INVESTMENT:,} in {STOCK}")
print(f"[{x}]  [{x:.2f}]  [{x:.1%}]  [{x:+.2%}]  [{x:>9.1%}]")

d = pd.Timestamp('1995-06-30')
print(f"{d:%Y-%m}   {d:%B %Y}")

### 3. Calling a function: `yf.download`, `.iloc` <a id="yfinance"></a>

> ```python
> px  = yf.download(STOCK, start=BIRTH_DATE, end=END_DATE, progress=False, auto_adjust=True)['Close']
> tot = float(px.iloc[-1] / px.iloc[0]) - 1
> ```

A function takes inputs in parentheses and hands back a result. Inputs come in
two ways: by **position** — `STOCK` comes first, so `yf.download` knows it is
the ticker — and by **keyword**, like `start=...` and `end=...`. A keyword you
leave out takes a default value.

That default is what the Live Demo's check cell is about. `auto_adjust`
decides whether the prices include dividends. If the AI's code leaves it out,
you get whatever the library's default happens to be. Two calls that differ in
one keyword give different answers, and both run without complaint.

`['Close']` picks one column out of what came back. `.iloc` picks rows by
**position**: `.iloc[0]` is the first row, `.iloc[-1]` the last (negative
positions count from the end). So `px.iloc[-1] / px.iloc[0] - 1` is last price
over first price, minus one: the return over the whole window. `float(...)`
turns the result into a plain number.

The check cell wraps all of this in `try:` … `except Exception:`. If anything
inside `try` fails — Yahoo is down, no internet — Python runs the `except` part
instead of stopping with an error.

📍 **Used in:** [Live Demo, Step 2 — the 🔒 Check cell](https://amoreira2.github.io/UG54/chapters/Finance/L1_Welcome_Returns_AI.html#step-2-implement).

In [ ]:
# four month-end prices, made up
px = pd.Series([100.0, 104.0, 98.0, 120.0],
               index=pd.to_datetime(['2020-01-31', '2020-02-29', '2020-03-31', '2020-04-30']))
print(px, "\n")

print("first :", px.iloc[0])
print("last  :", px.iloc[-1])
print(f"return: {px.iloc[-1] / px.iloc[0] - 1:.1%}")

### 4. Fama-French from the internet: `web.DataReader` <a id="datareader"></a>

> ```python
> import pandas_datareader.data as web
> ff = web.DataReader('F-F_Research_Data_Factors', 'famafrench', start='1980-01-01')[0]
> ```

`pandas_datareader` downloads data from public sources straight into pandas.
The source here is `'famafrench'`, Ken French's data library, and the dataset
is his monthly factors — which include the risk-free rate `RF` and the
market's excess return `Mkt-RF`.

What comes back is not one table but a **dictionary** of them: labelled boxes.
Box `0` holds the monthly table, box `1` the annual one, and `'DESCR'` a text
description. The `[0]` at the end of the line opens box 0.

📍 **Used in:** [Excess Returns and the Units Trap](https://amoreira2.github.io/UG54/chapters/Finance/L1_Welcome_Returns_AI.html#excess); again in
[*The Panel and Portfolio Mathematics*](https://amoreira2.github.io/UG54/chapters/Finance/L2_Panel_Portfolios_AI.html#but-ge-is-one-company) and its Challenge.

In [ ]:
import pandas_datareader.data as web

raw = web.DataReader('F-F_Research_Data_Factors', 'famafrench', start='1980-01-01')
print(type(raw).__name__, "with keys", list(raw.keys()), "\n")

ff = raw[0]          # box 0: the monthly table
ff.head(3)

### 5. Fixing the dates: `to_timestamp` and `MonthEnd` <a id="monthend"></a>

> ```python
> ff.index = pd.to_datetime(ff.index.to_timestamp()) + pd.offsets.MonthEnd(0)
> ff = ff / 100
> ```

The most cryptic line so far. It changes the **index** — the row labels,
printed down the left of the table — and nothing else.

Ken French labels each row with a *month*, `1980-01`, which pandas stores as a
`Period`. The course panel labels each row with a *date*, the last day of the
month: `1980-01-31`. Two tables only line up when their labels are the same
kind of thing (entry 10 shows why), so the line converts one into the other:

1. `.to_timestamp()` turns each month into a date — its **first** day,
   `1980-01-01`.
2. `+ pd.offsets.MonthEnd(0)` rolls each date forward to the end of its month.
   The `0` means "if it is already a month-end, leave it there".

The `pd.to_datetime(...)` wrapped around step 1 makes sure pandas treats the
result as dates. Here it changes nothing you can see.

Then `/ 100`. Ken French reports percent, so `0.65` means 0.65%. Dividing the
whole table by 100 puts it in decimals, the same units as `ret` in the panel.
Leaving it out is the units trap from the lecture.

📍 **Used in:** [Excess Returns and the Units Trap](https://amoreira2.github.io/UG54/chapters/Finance/L1_Welcome_Returns_AI.html#the-trap), and every
notebook that loads `ff`.

In [ ]:
idx = ff.index[:3]
print("as delivered      :", list(idx.astype(str)), "  a", type(idx).__name__)

step1 = idx.to_timestamp()
print("1. to_timestamp() :", list(step1.strftime('%Y-%m-%d')))

step2 = pd.to_datetime(step1) + pd.offsets.MonthEnd(0)
print("2. + MonthEnd(0)  :", list(step2.strftime('%Y-%m-%d')))

# the line from the lecture, then the units
ff.index = pd.to_datetime(ff.index.to_timestamp()) + pd.offsets.MonthEnd(0)
ff = ff / 100
ff.head(3)

### 6. Columns: `ff['RF']` and `ff[['Mkt-RF', 'RF']]` <a id="columns"></a>

> ```python
> print(ff[['Mkt-RF', 'RF']].head(3).to_string())
> print(f"mean RF = {ff['RF'].mean():.4f}")
> ```

A pandas table is a **DataFrame**. Each column on its own is a **Series**: one
column of values with the row labels attached.

- **One** name in brackets, `ff['RF']`, gives a Series.
- A **list** of names, `ff[['Mkt-RF', 'RF']]`, gives a smaller DataFrame. The
  outer brackets select; the inner ones make the list.

`.head(3)` keeps the first three rows. `.to_string()` turns the table into
plain text, so `print` shows all of it neatly. `.mean()` averages a column;
`.std()`, `.min()` and `.max()` work the same way.

You will also see `panel.permno` in place of `panel['permno']`. It is the same
column. The dot is shorter, but it only works when the name has no spaces or
symbols — `ff.Mkt-RF` would be read as `ff.Mkt` minus `RF` — and it cannot
create a new column.

📍 **Used in:** [Excess Returns and the Units Trap](https://amoreira2.github.io/UG54/chapters/Finance/L1_Welcome_Returns_AI.html#the-trap).

In [ ]:
rf = ff['RF']
print(type(rf).__name__)
print(rf.head(3), "\n")

two = ff[['Mkt-RF', 'RF']]
print(type(two).__name__)
print(two.head(3).to_string(), "\n")

print(f"mean RF = {ff['RF'].mean():.4f}   per month, in decimals")

### 7. The course panel: `pd.read_parquet` <a id="parquet"></a>

> ```python
> URL = ("https://raw.githubusercontent.com/amoreira2/UG54/refs/heads/main/assets/data/panel_backbone_1980_2000.parquet")
> panel = pd.read_parquet(URL)
> ```

Parquet is a file format for tables. It is smaller and much faster to load than
a CSV or a spreadsheet, and it remembers each column's type, so dates arrive as
dates. `pd.read_parquet` reads one from a file or, as here, straight from a web
address.

The parentheses around the address let it be split across lines. Python glues
two strings that sit side by side into one — `"abc" "def"` is `"abcdef"` — and
the Challenge solution uses that to keep the line short.

Then the first look at what you loaded:

- `len(panel)` — the number of rows
- `panel.permno.nunique()` — the number of *distinct* values in a column
- `panel.date.min()`, `.max()` — the first and last date; `.date()` drops the
  time of day for printing
- `list(panel.columns)` — the column names; `panel.head()` — the first five rows

📍 **Used in:** [Challenge: General Electric vs the Market](https://amoreira2.github.io/UG54/chapters/Finance/L1_Welcome_Returns_AI.html#challenge-general-electric-vs-the-market);
again at the top of [The Stacked Panel](https://amoreira2.github.io/UG54/chapters/Finance/L2_Panel_Portfolios_AI.html#the-stacked-panel).

In [ ]:
URL = ("https://raw.githubusercontent.com/amoreira2/UG54/"
       "refs/heads/main/assets/data/panel_backbone_1980_2000.parquet")
panel = pd.read_parquet(URL)

print(f"{len(panel):,} rows | {panel.permno.nunique():,} stocks | {panel.date.nunique()} months")
print(f"{panel.date.min().date()} to {panel.date.max().date()}")
print("columns:", list(panel.columns))
panel.head()

### 8. One stock out of the panel: masks and `set_index` <a id="onestock"></a>

> ```python
> ge = panel[panel.permno == 12060].set_index('date')['ret'].sort_index()
> ```

Read a chain like this left to right. Each step hands its result to the next.

1. `panel.permno == 12060` compares every row's `permno` with 12060 and gives
   a column of `True`/`False` — a **mask**. (`==` compares; a single `=` would
   store.)
2. `panel[mask]` keeps the rows where the mask is `True`: GE's 252 months.
3. `.set_index('date')` makes the date the row label, so each return is
   labelled by its month.
4. `['ret']` keeps just the return column, as a Series.
5. `.sort_index()` puts it in date order.

📍 **Used in:** [Challenge Solutions](https://amoreira2.github.io/UG54/chapters/Finance/L1_Welcome_Returns_AI.html#solutions); again in
[What Returns Actually Look Like](https://amoreira2.github.io/UG54/chapters/Finance/L2_Panel_Portfolios_AI.html#distributions) and
[Timing is everything](https://amoreira2.github.io/UG54/chapters/Finance/L2_Panel_Portfolios_AI.html#timing-is-everything).

In [ ]:
mask = panel.permno == 12060
print(mask.head(3), "\n")
print(f"{mask.sum()} rows are True\n")        # True counts as 1 when you add

ge = panel[mask].set_index('date')['ret'].sort_index()
print(ge.head(3))

### 9. Whole-column arithmetic: `.prod()`, `.mean()`, `.std()` <a id="vector"></a>

> ```python
> ge_total_return = (1 + ge).prod() - 1
> ge_ann_excess   = ge_excess.mean() * 12
> ge_ann_vol      = ge_excess.std()  * np.sqrt(12)
> ```

Arithmetic on a Series happens to every element at once. `1 + ge` adds one to
each of GE's 252 returns, with no loop. Then:

- `.prod()` multiplies all the elements together, so `(1 + ge).prod() - 1` is
  the compounded return.
- `.sum()` adds them up. Summing returns is pitfall 4 in the lecture's
  checklist: it runs, and it answers a different question.
- `.cumprod()` is the running product — the value of $1 month by month rather
  than only at the end. Growth-of-$1 plots draw it (entry 21).
- `.mean()` and `.std()` are the average and the standard deviation. Times 12
  and times `np.sqrt(12)`, they turn monthly numbers into annual ones.

📍 **Used in:** the [Pitfall Checklist for Returns](https://amoreira2.github.io/UG54/chapters/Finance/L1_Welcome_Returns_AI.html#pitfalls) and the
[Challenge Solutions](https://amoreira2.github.io/UG54/chapters/Finance/L1_Welcome_Returns_AI.html#solutions).

In [ ]:
r = pd.Series([0.10, -0.10, 0.10, -0.10])       # four months: up, down, up, down

print("1 + r      :", (1 + r).tolist())
print(f"sum        : {r.sum():+.4f}")
print(f"compounded : {(1 + r).prod() - 1:+.4f}")
print("cumprod    :", (1 + r).cumprod().round(4).tolist())

### 10. Lining up two series by date: `.reindex`, `.dropna` <a id="align"></a>

> ```python
> rf        = ff['RF'].reindex(ge.index)
> ge_excess = (ge - rf).dropna()
> ```

When you subtract one Series from another, pandas pairs them by **label**, not
by position: GE's June 1995 return meets June 1995's risk-free rate, wherever
each one sits. A label found in only one of the two gives `NaN` — "not a
number", pandas' marker for a missing value.

`.reindex(ge.index)` makes the pairing explicit. It returns `RF` at exactly
GE's dates, in GE's order, with `NaN` for any date Ken French doesn't have.
`.dropna()` then drops the rows that are missing.

This is why entry 5 fixed the dates. Had `ff` still been labelled `1980-01`
and the panel `1980-01-31`, no label would match: every excess return would be
`NaN`, with no error to tell you.

📍 **Used in:** [Challenge Solutions](https://amoreira2.github.io/UG54/chapters/Finance/L1_Welcome_Returns_AI.html#solutions), Q2 and Q3.

In [ ]:
a = pd.Series([0.05, 0.02, -0.01],
              index=pd.to_datetime(['1995-04-30', '1995-05-31', '1995-06-30']))
b = pd.Series([0.004, 0.005],                          # other order, April missing
              index=pd.to_datetime(['1995-06-30', '1995-05-31']))

print(a - b, "\n")             # paired by date, not by position
print((a - b).dropna(), "\n")
print(b.reindex(a.index))      # b, at a's dates

### 11. The memo and the submission cell <a id="submission"></a>

> ```python
> MEMO = """
> Write your memo here. Don't delete the surrounding triple quotes.
> """
> ```

Triple quotes make a string that can run over several lines, which is why the
memo sits inside them. Delete one of the quotes and Python can no longer tell
where the text ends, so the cell stops with a `SyntaxError`.

The submission cell is one you run but never edit. It does three things:

1. `missing = [v for v in required if v not in globals()]` is a **list
   comprehension**: go through each name `v` in `required`, and keep the ones
   the notebook doesn't know. `globals()` is everything the notebook has in
   memory right now. So a variable counts only if the cell that defines it has
   been **run** in this session. Typing it isn't enough, and restarting the
   runtime forgets everything.
2. If anything is missing, `raise NameError(...)` stops with a message that
   lists it.
3. Otherwise it packs your answers and memo into one line of text: the token
   you paste into the form.

📍 **Used in:** [Submission](https://amoreira2.github.io/UG54/chapters/Finance/L1_Welcome_Returns_AI.html#submit), at the end of every lecture.

In [ ]:
x = 1
MEMO = """
Two lines
of memo.
"""
required = ['x', 'MEMO', 'a_name_never_defined']

missing = [v for v in required if v not in globals()]
print("missing:", missing, "\n")

print(repr(MEMO))              # repr shows the line breaks as \n
print(repr(MEMO.strip()))      # .strip() trims them from both ends

---

## From *The Panel and Portfolio Mathematics*

### 12. The panel is long <a id="long"></a>

> ```python
> print(f"average of {len(panel)/panel.date.nunique():.0f} stocks per month")
> ```

The panel has one row per stock per month, stacked. Almost every question in
the course starts from one of two slices of it:

- **Fix a stock, follow it through time** — a time series.
  `panel[panel.permno == 12060]`, entry 8.
- **Fix a month, look across stocks** — a cross-section.
  `panel[panel.date == '1995-06-30']`.

The second filter compares dates with a string. pandas reads `'1995-06-30'` as
a date for the comparison, so you don't have to build one.

Most of the code in this lecture takes the second slice for *every* month at
once. That is what `groupby` is for (entries 15 and 20).

📍 **Used in:** [The Stacked Panel](https://amoreira2.github.io/UG54/chapters/Finance/L2_Panel_Portfolios_AI.html#the-stacked-panel) and
[Long, not wide](https://amoreira2.github.io/UG54/chapters/Finance/L2_Panel_Portfolios_AI.html#long-not-wide).

In [ ]:
june = panel[panel.date == '1995-06-30']
print(f"June 1995: {len(june):,} stocks\n")
print(june.head(3).to_string(index=False), "\n")

print(f"average of {len(panel)/panel.date.nunique():.0f} stocks per month")

### 13. `scipy.stats`: skewness and kurtosis <a id="scipy"></a>

> ```python
> from scipy import stats
> print(f"skewness {stats.skew(ge):.2f}   excess kurtosis {stats.kurtosis(ge):.2f}")
> ```

`scipy` is a fourth library, for statistics. `from scipy import stats` loads
just its `stats` part, so you write `stats.skew(...)` rather than
`scipy.stats.skew(...)`.

- `stats.skew(x)` measures asymmetry. Positive means a long right tail: rare,
  large gains.
- `stats.kurtosis(x)` measures how fat the tails are. By default it reports
  **excess** kurtosis, with 3 already subtracted, so a normal distribution
  scores 0. That default is why the lecture can print "(normal: 0)" beside it.
- `stats.norm.pdf(t)` is the height of the bell curve at each point in `t`:
  the dashed line in the lecture's histograms (entry 14).

One missing value makes these return `nan`, which is why the lecture's GE line
ends in `.dropna()`.

📍 **Used in:** [What Returns Actually Look Like](https://amoreira2.github.io/UG54/chapters/Finance/L2_Panel_Portfolios_AI.html#distributions).

In [ ]:
from scipy import stats

ge = panel[panel.permno == 12060].set_index('date')['ret'].dropna()      # GE, 252 months

print(f"skewness          {stats.skew(ge):6.2f}")
print(f"excess kurtosis   {stats.kurtosis(ge):6.2f}    the default: 3 already subtracted")
print(f"kurtosis          {stats.kurtosis(ge, fisher=False):6.2f}    fisher=False: the raw number\n")

print("with a missing value:", stats.skew(pd.Series([0.10, np.nan, -0.20, 0.05])))

### 14. A figure: `fig, ax = plt.subplots()` <a id="figure"></a>

> ```python
> fig, ax = plt.subplots(figsize=(7.5, 3.4))
> ax.hist(z, bins=40, density=True, alpha=0.65, color='steelblue', label='GE')
> ax.plot(t, stats.norm.pdf(t), 'k--', lw=1.6, label='Normal')
> ax.set_xlabel('standard deviations from the mean'); ax.legend()
> plt.tight_layout(); plt.show()
> ```

Every figure in the course is built the same way.

- `plt.subplots(...)` creates a figure and hands back two things: `fig`, the
  whole image, and `ax`, the plotting area inside it. `fig, ax = ...` stores
  them under two names at once. `figsize=(width, height)` is in inches.
- You draw on `ax`: `ax.hist(...)` for a histogram, `ax.plot(x, y)` for a line.
  `bins=40` is the number of bars. `density=True` scales the bars so their total
  area is one, which is what lets a histogram share an axis with a bell curve.
  `alpha` is transparency. `'k--'` means black (`k`), dashed (`--`), and `lw`
  is the line width.
- You label: `ax.set_xlabel`, `ax.set_title`, and `ax.legend()`, which lists
  every `label=` you gave.
- `plt.tight_layout(); plt.show()` tidies the margins and draws it. The
  semicolon lets two statements share a line; it only saves space.

Two lines before the figure prepare the data. `z = (ge - ge.mean()) / ge.std()`
rescales the returns into standard deviations from their mean, and
`np.linspace(-4, 4, 300)` makes 300 evenly spaced points from −4 to 4 — the
x-values at which the bell curve is drawn.

📍 **Used in:** [What Returns Actually Look Like](https://amoreira2.github.io/UG54/chapters/Finance/L2_Panel_Portfolios_AI.html#distributions), and every
figure after it.

In [ ]:
z = (ge - ge.mean()) / ge.std()        # GE's returns, in standard deviations
t = np.linspace(-4, 4, 300)            # x-values for the bell curve

fig, ax = plt.subplots(figsize=(7.5, 3.4))
ax.hist(z, bins=40, density=True, alpha=0.65, color='steelblue', label='GE')
ax.plot(t, stats.norm.pdf(t), 'k--', lw=1.6, label='Normal')
ax.set_xlabel('standard deviations from the mean'); ax.legend()
ax.set_title('Now try bins=10, and bins=100', fontweight='bold')
plt.tight_layout(); plt.show()

### 15. `groupby`, part one: one number per stock <a id="groupby1"></a>

> ```python
> g   = panel.dropna(subset=['ret']).groupby('permno')['ret']
> n   = g.size()
> big = n[n >= 120].index
> per = pd.DataFrame({'vol':  g.std()[big] * np.sqrt(12),
>                     'skew': g.apply(stats.skew)[big],
>                     'kurt': g.apply(stats.kurtosis)[big]})
> print(f"{(per['skew'] > 0).mean():.0%} of stocks are positively skewed")
> ```

The first `groupby` in the course. `panel.groupby('permno')` sorts the rows
into one pile per stock, and `['ret']` says which column you care about.
Nothing is computed yet. Whatever you ask for next is done **once per pile**,
and the answers come back as one Series labelled by `permno`:

- `g.size()` — how many rows each stock has
- `g.std()` — each stock's standard deviation; `.mean()`, `.min()` and the rest
  work the same way
- `g.apply(stats.skew)` — for a function pandas doesn't have built in, `.apply`
  runs it on each pile

Before the `groupby`, `.dropna(subset=['ret'])` drops the rows whose `ret` is
missing — only those. A gap in some other column doesn't remove the row.

Three more pieces:

- `n[n >= 120]` is a mask on a Series (entry 8): the stocks with at least 120
  months. `.index` keeps just their labels, the permnos.
- `g.std()[big]` picks those permnos out by label.
- `pd.DataFrame({'vol': ..., 'skew': ...})` builds a table from named Series.
  The `{ }` is a **dictionary**: each name becomes a column, and the rows line
  up by permno.

Last, `(per['skew'] > 0).mean()`. The comparison gives `True`/`False`. `True`
counts as 1 and `False` as 0, so the mean is the **share** that is true.

📍 **Used in:** [But GE is one company](https://amoreira2.github.io/UG54/chapters/Finance/L2_Panel_Portfolios_AI.html#but-ge-is-one-company).

In [ ]:
toy = pd.DataFrame({'permno': [1, 1, 1, 2, 2, 3],
                    'ret':    [0.10, -0.05, 0.02, 0.30, np.nan, -0.20]})
print(toy.to_string(index=False), "\n")

g = toy.dropna(subset=['ret']).groupby('permno')['ret']
n = g.size()
print("size per stock:\n" + n.to_string(), "\n")
print("mean per stock:\n" + g.mean().to_string(), "\n")

big = n[n >= 2].index
print("stocks with 2+ returns:", list(big))
print(f"share of stocks with a positive mean: {(g.mean() > 0).mean():.0%}")

### 16. `.rename`, `.idxmin`, `.strftime` <a id="idxmin"></a>

> ```python
> mkt   = (ff['Mkt-RF'] + ff['RF']).rename('ff_mkt')
> worst = mkt.idxmin().strftime('%B %Y')
> ```

- `ff['Mkt-RF'] + ff['RF']` adds the risk-free rate back to the market's
  excess return, date by date (entry 10). The result is the market's return.
- `.rename('ff_mkt')` gives the Series a name, which shows up as the column
  header if you later put it in a table.
- `.min()` is the lowest value. `.idxmin()` is its **label** — here, the date
  it happened. `.idxmax()` does the same for the highest.
- `.strftime('%B %Y')` writes a date as text in the format you ask for, with
  the codes from entry 2.

📍 **Used in:** [But GE is one company](https://amoreira2.github.io/UG54/chapters/Finance/L2_Panel_Portfolios_AI.html#but-ge-is-one-company), for the
market's worst month; `.rename('ff_mkt')` again in the
[Challenge](https://amoreira2.github.io/UG54/chapters/Finance/L2_Panel_Portfolios_AI.html#challenge-rebuild-the-market).

In [ ]:
mkt = (ff['Mkt-RF'] + ff['RF']).rename('ff_mkt')
print(mkt.head(3), "\n")

print(f"lowest value : {mkt.min():.1%}")
print(f"its label    : {mkt.idxmin()}")
print(f"as text      : {mkt.idxmin().strftime('%B %Y')}")

### 17. `.shift(1)`: last month's value <a id="shift"></a>

> ```python
> ge = panel[panel.permno == 12060].head(4)[['date', 'ret', 'me']]
> ge['me_l1'] = ge['me'].shift(1)
> ```

`.shift(1)` moves a column down one row. Each row now holds the value from the
row above, and the first row, with nothing above it, gets `NaN`. Next to this
month's `ret`, that is last month's `me` — the market cap you knew when you
decided what to hold. `.shift(-1)` goes the other way and pulls the next row's
value up.

`ge['me_l1'] = ...` creates a new column by assigning to a name that doesn't
exist yet. This is where the dot shortcut from entry 6 doesn't work.

In the same cell, `.head(4)` keeps the first four rows, and
`.to_string(index=False)` prints without the row labels.

📍 **Used in:** [Timing is everything](https://amoreira2.github.io/UG54/chapters/Finance/L2_Panel_Portfolios_AI.html#timing-is-everything).

In [ ]:
ge4 = panel[panel.permno == 12060].head(4)[['date', 'ret', 'me']]   # ge4, so entry 13's ge survives
ge4['me_l1']    = ge4['me'].shift(1)
ge4['ret_next'] = ge4['ret'].shift(-1)
print(ge4.to_string(index=False))

### 18. `groupby('permno')['me'].shift(1)`: last month's value, per stock <a id="groupshift"></a>

> ```python
> df['me_l1'] = df.groupby('permno')['me'].shift(1)   # ✅
> df['me_l1'] = df['me'].shift(1)                     # ❌ crosses stocks
> ```

On the whole panel, a plain `.shift(1)` goes wrong. The panel is sorted by
stock and then by date, so the row above a stock's first month is the
*previous stock's last month*. The ❌ line gives every stock, in its first
month, somebody else's market cap. It runs without complaint, and on the
course panel it is wrong in 17,799 rows — every stock but the first.

`groupby('permno')` fixes it. The shift happens inside each stock's pile, so
each stock's first month gets `NaN`.

There are two things `.shift` doesn't know:

- **Order.** It uses the rows in whatever order they are in. The panel comes
  sorted by `permno` and then `date`. If you ever re-sort it, put it back with
  `.sort_values(['permno', 'date'])` before you shift.
- **The calendar.** It moves one *row*, not one *month*. If a stock's history
  has a gap, the row above can be several months back; the panel has 1,357
  such rows. The lecture's 🔒 Check cell guards against this. It computes the
  date one month on with `+ pd.offsets.MonthEnd(1)` — the `1` means "the next
  month-end", where entry 5's `0` meant "this one" — and `.where(ok)` keeps a
  value only where that check is `True`, putting `NaN` everywhere else.

`.reset_index(drop=True)`, in the same Check cell, renumbers the rows 0, 1, 2,
… after sorting and throws the old numbering away.

📍 **Used in:** [build it yourself for the entire data set](https://amoreira2.github.io/UG54/chapters/Finance/L2_Panel_Portfolios_AI.html#build-it-yourself-for-the-entire-data-set)
and [Two answers, 14× apart](https://amoreira2.github.io/UG54/chapters/Finance/L2_Panel_Portfolios_AI.html#two-answers-14-apart). From here on, every
notebook starts by making `me_l1` this way.

In [ ]:
toy = pd.DataFrame({'permno': [1, 1, 1, 2, 2],
                    'date':   pd.to_datetime(['1995-01-31', '1995-02-28', '1995-03-31',
                                              '1995-01-31', '1995-02-28']),
                    'me':     [100, 110, 120, 5, 6]})

toy['plain']   = toy['me'].shift(1)                      # ❌ stock 2 gets stock 1's March
toy['grouped'] = toy.groupby('permno')['me'].shift(1)    # ✅
print(toy.to_string(index=False))

In [ ]:
# A stock with a gap: March and April are missing
gap = pd.DataFrame({'permno': [1, 1, 1],
                    'date':   pd.to_datetime(['1995-01-31', '1995-02-28', '1995-05-31']),
                    'me':     [100, 110, 150]})

gap['me_l1'] = gap.groupby('permno')['me'].shift(1)
prev_date    = gap.groupby('permno')['date'].shift(1)
ok           = prev_date + pd.offsets.MonthEnd(1) == gap['date']    # is the row above really last month?
gap['me_l1_safe'] = gap['me_l1'].where(ok)
print(gap.to_string(index=False))

### 19. One month's portfolio by hand: weights, `@`, `np.average` <a id="weights"></a>

> ```python
> d = panel.dropna(subset=['ret', 'me_l1'])
> m = d[d.date == '1995-06-30'].copy()
> w = m['me_l1'] / m['me_l1'].sum()
> ```

- `dropna(subset=['ret', 'me_l1'])` keeps the rows that have both a return and
  last month's market cap: the stocks you could have held. It comes *before*
  the weights, so the weights sum to one over exactly those stocks.
- `.copy()` makes `m` a table of its own, so changing it later can't touch `d`.
- `m['me_l1'] / m['me_l1'].sum()` divides each stock's size by the total. Those
  are the value weights.

Three ways to get the portfolio return w′r, all giving the same number:

- `(w * m['ret']).sum()` — multiply stock by stock, then add up
- `w @ m['ret']` — `@` is multiply-and-add in one symbol: the w′r of the lecture
- `np.average(m['ret'], weights=m['me_l1'])` — does the dividing for you, so you
  pass raw market caps as `weights=`

The lecture's check line also uses `.loc`: `vw.loc['1995-06-30']` picks a value
by its **label**, where `.iloc` (entry 3) picks by **position**.

📍 **Used in:** [Two answers, 14× apart](https://amoreira2.github.io/UG54/chapters/Finance/L2_Panel_Portfolios_AI.html#two-answers-14-apart) and
[Step 3 — Validate](https://amoreira2.github.io/UG54/chapters/Finance/L2_Panel_Portfolios_AI.html#step-3-validate).

In [ ]:
panel['me_l1'] = panel.groupby('permno')['me'].shift(1)
d = panel.dropna(subset=['ret', 'me_l1'])

m = d[d.date == '1995-06-30'].copy()
w = m['me_l1'] / m['me_l1'].sum()

print(f"stocks: {len(m):,}   weights sum to {w.sum():.6f}\n")
print(f"(w * r).sum()  {(w * m['ret']).sum():.6f}")
print(f"w @ r          {w @ m['ret']:.6f}")
print(f"np.average     {np.average(m['ret'], weights=m['me_l1']):.6f}\n")

s = pd.Series([10, 20, 30], index=['a', 'b', 'c'])
print(".loc['b'] :", s.loc['b'], "   .iloc[1] :", s.iloc[1])

### 20. `groupby`, part two: one number per month, and `lambda` <a id="groupby2"></a>

> ```python
> vw = d.groupby('date').apply(lambda g: np.average(g['ret'], weights=g['me_l1']))
> ```

Entry 19 did one month. This line does all 251.

`d.groupby('date')` sorts the rows into one pile per month. When a built-in
like `.mean()` is enough, you pick a column and call it, as in entry 15. The
value-weighted return needs two columns at once, `ret` and `me_l1`, and there
is no built-in for that. So `.apply` hands each month's whole pile — a small
DataFrame — to a function you supply, and collects the answers into a Series
labelled by date.

The function here is a **lambda**: a one-line function with no name.
`lambda g: np.average(g['ret'], weights=g['me_l1'])` means "given a pile, call
it `g`, and return this". It is entry 19's calculation with `g` in place of
`m`. The name `g` is arbitrary; `lambda rows: ...` does the same thing.

The lecture's Appendix builds up to this line in four steps — a for loop,
`groupby`, a named `def` function, then the lambda — and checks that all four
agree. Read it if this line still looks like magic.

The equal-weighted version is the Hands-On. You have every piece you need
from entry 15.

📍 **Used in:** [Two answers, 14× apart](https://amoreira2.github.io/UG54/chapters/Finance/L2_Panel_Portfolios_AI.html#two-answers-14-apart), the
[Hands-On](https://amoreira2.github.io/UG54/chapters/Finance/L2_Panel_Portfolios_AI.html#ho1), and the
[Appendix on `groupby(...).apply`](https://amoreira2.github.io/UG54/chapters/Finance/L2_Panel_Portfolios_AI.html#groupby).

In [ ]:
vw = d.groupby('date').apply(lambda g: np.average(g['ret'], weights=g['me_l1']))

print(f"{len(vw)} months, one number each\n")
print(vw.head(3), "\n")
print(f"June 1995: {vw.loc['1995-06-30']:.6f}   <- entry 19's number")

### 21. Growth of $1 on a log scale <a id="logscale"></a>

> ```python
> ax.plot(vw.index, (1+vw).cumprod(), label='Value-weighted', linewidth=1.8)
> ax.set_yscale('log'); ax.set_ylabel('Growth of $1 (log scale)')
> ```

`(1 + vw).cumprod()` is entry 9's running product: what $1 invested at the
start is worth each month. `ax.plot(x, y)` draws it against the dates.

`ax.set_yscale('log')` puts the vertical axis on a log scale, where each step
up is a *multiplication*: 1, 10, 100. On that scale a steady 10% a year is a
straight line, and a 50% loss looks the same size at $2 as at $200. On an
ordinary scale the last few years of any long compounding series dwarf
everything before them.

📍 **Used in:** [Hands-On: Which One Is Bigger, and Why?](https://amoreira2.github.io/UG54/chapters/Finance/L2_Panel_Portfolios_AI.html#ho1).

In [ ]:
ge  = panel[panel.permno == 12060].set_index('date')['ret']
mkt = (ff['Mkt-RF'] + ff['RF']).reindex(ge.index)       # the market, at GE's dates (entry 10)

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(ge.index,  (1 + ge).cumprod(),  label='GE',     linewidth=1.8)
ax.plot(mkt.index, (1 + mkt).cumprod(), label='Market', linewidth=1.8)
ax.set_yscale('log'); ax.set_ylabel('Growth of $1 (log scale)')
ax.set_title('Delete set_yscale and re-run', fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()

---

*Entries for* Sorts, Breakpoints, and Long-Short Portfolios *are added after
that lecture.*